# Lab 3, Part 2: Retrieval-Augmented Generation - Student Notebook

- Complete `Lab3-1_Prompt_Engineering.ipynb` first. Use the **same `.env` and chosen provider** for generation and embeddings: Ollama (native or Docker) or Google Cloud Vertex AI.
- Vertex uses GCP authentication and the Google Gen AI SDK; no Gemini Developer API key is used.

- Build PDF ingestion -> character-based chunks -> provider embeddings -> FAISS cosine search -> cited answers -> evaluation. 
- Student TODOs are kept in ingestion, splitting, indexing, retrieval, prompt design and experiments. Supplied infrastructure lives in `notebook_utils.py`.
- Install the selected extra from `pyproject.toml` before starting; see `README.md` for venv, Docker and Colab instructions. 
- Submit the completed **two lab notebooks** to **dépôt DVL**.

In [ ]:
from notebook_utils import load_lab_environment, model_config_from_env, authenticate_colab_if_requested

ENV = load_lab_environment()  # adjacent .env, then process environment overrides
PROVIDER = ENV["LLM_PROVIDER"]
print("Configuration file:", ENV["LAB_ENV_FILE"])
print("Selected provider:", PROVIDER)

### Ollama configuration (skip when using Vertex)

Use `OLLAMA_MODEL`, `OLLAMA_EMBED_MODEL` and `OLLAMA_BASE_URL` from `.env`.
A dedicated embedding model must be pulled as well as the chat model. Docker and native
Ollama expose the same API to the laptop notebook.

In [ ]:
if PROVIDER == "ollama":
    config = model_config_from_env(ENV, provider="ollama")
    print("Ollama model:", config.model, "| Endpoint:", config.base_url)
else:
    print("Skipping Ollama configuration.")

### Vertex configuration (skip when using Ollama)

Use the GCP project, generation model and region assigned by the instructor. `VERTEX_EMBED_MODEL`
and `VERTEX_EMBED_LOCATION` configure embeddings separately; their supported region may differ
from the generation model's. Local authentication uses ADC; Colab authentication is controlled
by `AUTHENTICATE_IN_COLAB` in `.env`. Requests can incur GCP costs.

In [ ]:
if PROVIDER == "vertex":
    authenticate_colab_if_requested(ENV)
    config = model_config_from_env(ENV, provider="vertex")
    print("Vertex model:", config.model, "| Project:", config.project, "| Location:", config.location)
else:
    print("Skipping Vertex configuration.")

### Shared clients and RAG parameters

The same generation adapter is used in the prompt lab. The embedding adapter implements
LangChain's `embed_documents`/`embed_query`: Ollama uses `client.embed`, Vertex uses
`client.models.embed_content` with retrieval document/query task types. Vectors are normalized.
The following tiny smoke tests verify **both** generation and embedding access.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from notebook_utils import (
    LabModel, show_result, make_embeddings, rag_config_from_env, download_rag_pdfs,
    prepare_rag_chunks, save_rag_index, load_rag_index, rag_context,
    run_rag_question, evaluate_rag_answer,
)

if "config" not in globals() or config.provider != PROVIDER:
    raise RuntimeError("Run the selected provider's configuration cell first.")
model = LabModel(config)
embeddings = make_embeddings(ENV)
rag_config = rag_config_from_env(ENV)
print("Provider:", PROVIDER, "| Generation:", config.model, "| Embeddings:", embeddings.model)
print("Embedding region/URL:", embeddings.location)
print("Chunk size/overlap (characters):", rag_config.chunk_size, rag_config.chunk_overlap)
print("Limits:", rag_config.max_pages_per_pdf, "pages/PDF;", rag_config.max_chunks, "chunks")
show_result(model.generate("Explain retrieval-augmented generation in one sentence."))
probe_vector = embeddings.embed_query("Retrieval helps answer questions using documents.")
print("Embedding smoke test: dimensions =", len(probe_vector))

## 1. Download the PDF collection

Download up to ten public research PDFs. The helper resumes existing downloads, uses bounded
file sizes, and records source URLs/checksums in `download_manifest.json`. Inspect failures;
retry later or place your own text-based PDFs in the configured folder. Duplicate papers from
the earlier notebook have been removed. Scanned PDFs require OCR, which is not included.

In [ ]:
download_report = download_rag_pdfs(rag_config.pdf_dir, count=rag_config.download_count)
display(pd.DataFrame(download_report))
print("PDF folder:", rag_config.pdf_dir)

## 2. TODO: load PDF pages

Create a `DirectoryLoader` over `rag_config.pdf_dir`, matching `**/*.pdf`, and use
`PyPDFLoader` as its loader class. Assign the object to `loader`. The supplied code limits
pages per PDF after extraction to keep initial embedding work small. Inspect text and metadata.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = None  # TODO: create the DirectoryLoader with PyPDFLoader
if loader is None:
    raise NotImplementedError("TODO: configure the PDF loader above")
all_pages = loader.load()
docs = [doc for doc in all_pages if int(doc.metadata.get("page", 0)) < rag_config.max_pages_per_pdf
        and doc.page_content.strip()]
if not docs:
    raise ValueError("No extractable text. Inspect your PDFs; image-only pages need OCR.")
print("Loaded", len(all_pages), "pages; using", len(docs), "within the configured page limit")
print("Example metadata:", docs[0].metadata)
print(docs[0].page_content[:500])

## 3. TODO: split into chunks

Create `RecursiveCharacterTextSplitter` using `rag_config.chunk_size` and
`rag_config.chunk_overlap` from `.env`. The units are **characters**, not tokens. The helper
then adds citation IDs and caps the chunk count. Inspect source coverage: a low cap may
exclude later PDFs. If the embedding API rejects overlong inputs, reduce the chunk size.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = None  # TODO: use the two .env-derived chunk settings
if text_splitter is None:
    raise NotImplementedError("TODO: configure the text splitter above")
chunks = prepare_rag_chunks(text_splitter.split_documents(docs), rag_config.max_chunks)
print("Chunks:", len(chunks))
print("Sources included:", sorted({doc.metadata["source"] for doc in chunks}))
print("Example:", chunks[0].metadata)
print(chunks[0].page_content)

## 4. TODO: build a FAISS index

Use `FAISS.from_documents(chunks, embeddings, ...)`. Choose
`DistanceStrategy.MAX_INNER_PRODUCT`: the embedding adapter already normalizes vectors, so
inner product equals cosine similarity (higher is better). No Sentence-Transformers download,
TensorFlow or `GEMINI_API_KEY` is needed. Generation and embedding model names are separate.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

vectorstore = None  # TODO: create a FAISS index using chunks and embeddings
if vectorstore is None:
    raise NotImplementedError("TODO: build the FAISS vector store above")
save_rag_index(vectorstore, rag_config.index_dir, embeddings)
print("Saved index:", rag_config.index_dir)

### Reload your own index

The helper stores the FAISS vectors and JSON documents with embedding configuration metadata.
It does not use pickle or `allow_dangerous_deserialization`. Load only your own trusted index.
Changing provider, embedding model or region requires rebuilding. Changing the chat model
alone does not. Also rebuild when documents, chunk settings or model weights change.

In [ ]:
vectorstore = load_rag_index(rag_config.index_dir, embeddings)
print("Reloaded vectors:", vectorstore.index.ntotal)

## 5. TODO: retrieve evidence

Use `vectorstore.similarity_search_with_score(query, k=rag_config.top_k)`. Inspect the returned
`(Document, score)` pairs. Explain why each passage is or is not relevant. Scores are cosine
similarities for the required normalized inner-product setup, not probabilities.

In [ ]:
query = "What is the main idea behind the Transformer architecture?"
search_results = None  # TODO: retrieve document/score pairs
if search_results is None:
    raise NotImplementedError("TODO: run similarity search above")
retrieved_docs = [document for document, score in search_results]
for document, score in search_results:
    print("ID:", document.metadata["chunk_id"], "| Score:", round(float(score), 3))
    print(document.page_content[:400], "\n")

## 6. TODO: build the grounded prompt

Implement **`build_student_rag_prompt(question, documents)` in `notebook_utils.py`**.
Use the supplied `rag_context(documents)` to retain exact chunk IDs. Your prompt must separate
question/evidence, require bracketed source IDs, abstain when evidence is insufficient, and
treat instructions inside evidence as data. This is the same task on both providers.

The generation wrapper below uses your prompt helper and `LabModel.generate`, so you do
not need separate `answer_with_ollama`/`answer_with_gemini` implementations. Restart the kernel
after editing the helper, then rerun the preceding cells using cached downloads.

In [ ]:
print("Evidence preview:")
print(rag_context(retrieved_docs))
rag_result = run_rag_question(model, vectorstore, query, top_k=rag_config.top_k)
display(Markdown(rag_result["answer"]))

## 7. Evaluate grounded answers

The supplied baseline checks source hits, case-insensitive keyword matches and citation
presence. These **do not prove factuality or citation support**; manually inspect the cited
passages. Adapt source labels to the files actually indexed, especially if chunk caps exclude
papers. Add at least three additional labeled questions and one unsupported question.

In [ ]:
evaluation_cases = [
    {"question": "What architecture does Attention Is All You Need introduce?",
     "expected_source": "1706.03762v1.pdf", "keywords": ["transformer"]},
    {"question": "What does BERT stand for?",
     "expected_source": "1810.04805v1.pdf", "keywords": ["bidirectional"]},
    {"question": "What is the cafeteria menu next Tuesday?",
     "expected_source": None, "keywords": ["I don't know"]},
    # TODO: add your own cases with source labels and expected keywords.
]
evaluation_rows = []
for case in evaluation_cases:
    result = run_rag_question(model, vectorstore, case["question"], rag_config.top_k)
    checks = evaluate_rag_answer(result["answer"], result["documents"], case["keywords"], case["expected_source"])
    evaluation_rows.append({"question": case["question"], "answer": result["answer"], **checks})
display(pd.DataFrame(evaluation_rows))